<a href="https://colab.research.google.com/github/llazdll/New-York-City-Taxi-Trip-Duration/blob/main/ensumbles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(f"GPU: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

GPU: True
Device: Tesla T4


In [ ]:
# ============================================================
# CELL 1 — Install packages
# ============================================================
!pip install -q kagglehub xgboost lightgbm catboost
print("✅ Packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00
✅ Packages installed


In [ ]:
# ============================================================
# CELL 2A — kagglehub login (zero upload)
# ============================================================
import kagglehub

# این یک URL چاپ می‌کند که باید باز کنی و کد را paste کنی
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
# ============================================================
# CELL 3 — Download NYC Taxi data (no upload)
# ============================================================
import os
import time

DATA_DIR = "/content/nyc_taxi"
os.makedirs(DATA_DIR, exist_ok=True)

# Try kagglehub first (no manual upload needed)
try:
    import kagglehub
    print("Downloading via kagglehub...")
    t0 = time.time()
    path = kagglehub.competition_download("nyc-taxi-trip-duration")
    print(f"  ✅ Downloaded in {time.time()-t0:.1f}s")
    print(f"  Path: {path}")
    os.system(f"cp {path}/* {DATA_DIR}/")
except Exception as e:
    print(f"  kagglehub failed: {e}")
    print("  Trying Kaggle CLI...")
    os.system(f"kaggle competitions download -c nyc-taxi-trip-duration -p {DATA_DIR}")
    os.system(f"cd {DATA_DIR} && unzip -o '*.zip'")

# Unzip if needed
if not os.path.exists(f"{DATA_DIR}/train.csv"):
    os.system(f"cd {DATA_DIR} && unzip -o '*.zip'")

# Verify
for f in ["train.csv", "test.csv"]:
    p = f"{DATA_DIR}/{f}"
    if os.path.exists(p):
        size_mb = os.path.getsize(p) / 1e6
        print(f"  ✅ {f}: {size_mb:.0f} MB")
    else:
        raise FileNotFoundError(f"Missing {p}")

print("\n✅ Data ready at", DATA_DIR)

100%|██████████| 85.8M/85.8M [00:00<00:00, 102MB/s]

Extracting files...


  ✅ Downloaded in 2.0s
  Path: /root/.cache/kagglehub/competitions/nyc-taxi-trip-duration
  ✅ train.csv: 201 MB
  ✅ test.csv: 71 MB

✅ Data ready at /content/nyc_taxi


In [ ]:
# ============================================================
# CELL 4 — Feature engineering in pandas
# Replicates the same 29 features as Spark pipeline
# ============================================================
import os
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

DATA_DIR = "/content/nyc_taxi"
SAVE_DIR = "/content/nyc_cache"
os.makedirs(SAVE_DIR, exist_ok=True)

def haversine_vec(lat1, lon1, lat2, lon2):
    """Vectorized haversine (km)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371 * np.arcsin(np.sqrt(a))


def build_features(df):
    """Same 29 features as Spark pipeline."""
    df = df.copy()
    df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])

    # ---- Temporal ----
    df["pickup_hour"]      = df["pickup_datetime"].dt.hour
    df["pickup_dayofweek"] = df["pickup_datetime"].dt.dayofweek + 1  # 1=Sun (pandas 0=Mon)
    df["pickup_month"]     = df["pickup_datetime"].dt.month

    df["is_weekend"]    = df["pickup_dayofweek"].isin([1, 7]).astype(int)
    df["is_rush_hour"]  = ((df["pickup_hour"].between(7, 9)) |
                            (df["pickup_hour"].between(17, 19))).astype(int)
    df["is_late_night"] = ((df["pickup_hour"] >= 22) | (df["pickup_hour"] <= 4)).astype(int)

    # ---- Cyclical ----
    df["pickup_hour_sin"]  = np.sin(2*np.pi * df["pickup_hour"] / 24)
    df["pickup_hour_cos"]  = np.cos(2*np.pi * df["pickup_hour"] / 24)
    df["pickup_dow_sin"]   = np.sin(2*np.pi * (df["pickup_dayofweek"]-1) / 7)
    df["pickup_dow_cos"]   = np.cos(2*np.pi * (df["pickup_dayofweek"]-1) / 7)
    df["pickup_month_sin"] = np.sin(2*np.pi * (df["pickup_month"]-1) / 12)
    df["pickup_month_cos"] = np.cos(2*np.pi * (df["pickup_month"]-1) / 12)

    # ---- Spatial ----
    df["haversine_km"] = haversine_vec(
        df["pickup_latitude"].values, df["pickup_longitude"].values,
        df["dropoff_latitude"].values, df["dropoff_longitude"].values,
    )
    df["distance_bin"] = np.floor(df["haversine_km"]).astype(int)

    df["is_jfk_pickup"] = (
        df["pickup_longitude"].between(-73.82, -73.74) &
        df["pickup_latitude"].between(40.62, 40.68)
    ).astype(int)
    df["is_lga_pickup"] = (
        df["pickup_longitude"].between(-73.90, -73.84) &
        df["pickup_latitude"].between(40.75, 40.80)
    ).astype(int)
    df["is_airport_pickup"] = np.maximum(df["is_jfk_pickup"], df["is_lga_pickup"])
    df["is_manhattan"] = (
        df["pickup_longitude"].between(-74.02, -73.90) &
        df["pickup_latitude"].between(40.70, 40.88)
    ).astype(int)
    df["slow_trip_indicator"] = ((df["is_rush_hour"] == 1) &
                                  (df["haversine_km"] < 3)).astype(int)

    # ---- Passenger ----
    df["passenger_count"] = df["passenger_count"].clip(upper=6)
    df["is_group_trip"] = (df["passenger_count"] >= 2).astype(int)

    # ---- store flag ----
    df["store_and_fwd_flag_y"] = (df["store_and_fwd_flag"] == "Y").astype(int)

    # ---- Interactions ----
    df["vendor_x_group"] = (df["vendor_id"].astype(str) + "_" +
                              df["is_group_trip"].astype(str))
    df["vendor_rush"] = ((df["vendor_id"] == 2) &
                          (df["is_rush_hour"] == 1)).astype(int)

    return df


FEATURE_COLS = [
    "vendor_id", "passenger_count",
    "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude",
    "store_and_fwd_flag_y",
    "pickup_hour", "pickup_dayofweek", "pickup_month",
    "is_weekend", "is_rush_hour", "is_late_night",
    "pickup_hour_sin", "pickup_hour_cos",
    "pickup_dow_sin",  "pickup_dow_cos",
    "pickup_month_sin","pickup_month_cos",
    "haversine_km", "distance_bin",
    "is_jfk_pickup", "is_lga_pickup", "is_airport_pickup",
    "is_manhattan", "slow_trip_indicator",
    "is_group_trip", "vendor_rush", "vendor_x_group_idx",
]

# ============================================================
# Load + process
# ============================================================
print("=== Loading CSVs ===")
t0 = time.time()
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(f"  train: {len(train):,} rows ({time.time()-t0:.1f}s)")
print(f"  test : {len(test):,} rows")

print("\n=== Building features ===")
t0 = time.time()
train = build_features(train)
test  = build_features(test)

# Label encode vendor_x_group (fit on train only)
le_vg = LabelEncoder()
le_vg.fit(train["vendor_x_group"])
train["vendor_x_group_idx"] = le_vg.transform(train["vendor_x_group"])
test["vendor_x_group_idx"]  = le_vg.transform(
    test["vendor_x_group"].map(lambda x: x if x in le_vg.classes_ else le_vg.classes_[0])
)
print(f"  done in {time.time()-t0:.1f}s")

# ============================================================
# Cleaning (train only) + time split
# ============================================================
print("\n=== Cleaning + split ===")
before = len(train)
train = train[
    (train["trip_duration"] > 0) &
    (train["trip_duration"].between(30, 86400)) &
    (train["passenger_count"] > 0) &
    (train["haversine_km"] > 0)
].copy()
print(f"  cleaned: {before:,} → {len(train):,}")

train["log_trip_duration"] = np.log1p(train["trip_duration"])

# Time split (same as Spark pipeline)
split_ts = pd.Timestamp("2016-06-01")
train_part = train[train["pickup_datetime"] < split_ts].copy()
val_part   = train[train["pickup_datetime"] >= split_ts].copy()
print(f"  train_part: {len(train_part):,}")
print(f"  val_part  : {len(val_part):,}")

# ============================================================
# Build numpy arrays
# ============================================================
print("\n=== Building numpy arrays ===")

def to_np(df, has_label=True):
    X = df[FEATURE_COLS].values.astype(np.float32)
    out = {"X": X}
    if has_label:
        out["y"]        = df["log_trip_duration"].values.astype(np.float32)
        out["y_sec"]    = df["trip_duration"].values.astype(np.float32)
    return out

train_np = to_np(train_part, True)
val_np   = to_np(val_part,   True)
test_np  = to_np(test,       False)

print(f"  X_train: {train_np['X'].shape}")
print(f"  X_val  : {val_np['X'].shape}")
print(f"  X_test : {test_np['X'].shape}")

# ============================================================
# Save to .npz (small — ~100 MB compressed, on Drive)
# ============================================================
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/nyc_taxi"
os.makedirs(DRIVE_DIR, exist_ok=True)

OUT = f"{DRIVE_DIR}/cache.npz"
print(f"\n=== Saving to {OUT} ===")
t0 = time.time()
np.savez_compressed(
    OUT,
    X_train=train_np["X"], y_train=train_np["y"], y_train_sec=train_np["y_sec"],
    X_val=val_np["X"],     y_val=val_np["y"],     y_val_sec=val_np["y_sec"],
    X_test=test_np["X"],
    test_ids=test["id"].values,
)
print(f"  ✅ Saved in {time.time()-t0:.1f}s | "
      f"size: {os.path.getsize(OUT)/1e6:.0f} MB")

print("\n✅ CELL 4 DONE")

=== Loading CSVs ===
  train: 1,458,644 rows (5.8s)
  test : 625,134 rows

=== Building features ===
  done in 6.1s

=== Cleaning + split ===
  cleaned: 1,458,644 → 1,449,273
  train_part: 1,216,551
  val_part  : 232,722

=== Building numpy arrays ===
  X_train: (1216551, 29)
  X_val  : (232722, 29)
  X_test : (625134, 29)
Mounted at /content/drive

=== Saving to /content/drive/MyDrive/nyc_taxi/cache.npz ===
  ✅ Saved in 28.5s | size: 50 MB

✅ CELL 4 DONE


In [16]:
# ============================================================
# CELL 5 — Neural Network (with resume from Drive)
# ============================================================
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

DRIVE_DIR = "/content/drive/MyDrive/nyc_taxi"
CACHE = f"{DRIVE_DIR}/cache.npz"
CKPT  = f"{DRIVE_DIR}/nn_checkpoint.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ---- Load ----
print("\n=== Loading cache ===")
data = np.load(CACHE, allow_pickle=True)
X_train     = data["X_train"]
X_val       = data["X_val"]
X_test      = data["X_test"]
y_train     = data["y_train"]
y_val       = data["y_val"]
y_val_sec   = data["y_val_sec"]
test_ids    = data["test_ids"]
print(f"  train: {X_train.shape} | val: {X_val.shape} | test: {X_test.shape}")

FEATURE_COLS = [
    "vendor_id", "passenger_count", "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude", "store_and_fwd_flag_y",
    "pickup_hour", "pickup_dayofweek", "pickup_month",
    "is_weekend", "is_rush_hour", "is_late_night",
    "pickup_hour_sin", "pickup_hour_cos",
    "pickup_dow_sin",  "pickup_dow_cos",
    "pickup_month_sin","pickup_month_cos",
    "haversine_km", "distance_bin",
    "is_jfk_pickup", "is_lga_pickup", "is_airport_pickup",
    "is_manhattan", "slow_trip_indicator",
    "is_group_trip", "vendor_rush", "vendor_x_group_idx",
]

# ---- Cat / cont split ----
cat_specs = {
    "vendor_id": 2, "passenger_count": 7, "pickup_hour": 24,
    "pickup_dayofweek": 7, "pickup_month": 6,
    "distance_bin": 31, "vendor_x_group_idx": 4,
}
cat_indices = [FEATURE_COLS.index(c) for c in cat_specs.keys()]
cont_indices = [i for i in range(len(FEATURE_COLS)) if i not in cat_indices]

def prep_cat(X):
    out = X[:, cat_indices].astype(np.int64).copy()
    for i, name in enumerate(cat_specs.keys()):
        if name in ("pickup_dayofweek", "pickup_month", "vendor_id"):
            out[:, i] -= 1
        out[:, i] = np.clip(out[:, i], 0, cat_specs[name] - 1)
    return out

print("\n=== Preprocessing ===")
X_train_cat = prep_cat(X_train); X_val_cat = prep_cat(X_val); X_test_cat = prep_cat(X_test)
X_train_cont = X_train[:, cont_indices].astype(np.float32)
X_val_cont   = X_val[:, cont_indices].astype(np.float32)
X_test_cont  = X_test[:, cont_indices].astype(np.float32)

scaler = StandardScaler()
X_train_cont = scaler.fit_transform(X_train_cont).astype(np.float32)
X_val_cont   = scaler.transform(X_val_cont).astype(np.float32)
X_test_cont  = scaler.transform(X_test_cont).astype(np.float32)
print(f"  cont: {X_train_cont.shape} | cat: {X_train_cat.shape}")

# ---- Model ----
class TabularNN(nn.Module):
    def __init__(self, cont_dim, cat_specs, hidden=(512, 256, 128), dropout=0.2):
        super().__init__()
        self.embs = nn.ModuleList()
        emb_total = 0
        for name, n in cat_specs.items():
            ed = min(16, max(4, (n + 1)//2))
            self.embs.append(nn.Embedding(n, ed))
            emb_total += ed
        layers = []
        prev = cont_dim + emb_total
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, cont, cat):
        embs = [emb(cat[:, i]) for i, emb in enumerate(self.embs)]
        x = torch.cat([cont] + embs, dim=-1)
        return self.net(x).squeeze(-1)

# ---- DataLoaders ----
BATCH_SIZE = 4096 if device.type == "cuda" else 1024
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train_cont), torch.from_numpy(X_train_cat),
                  torch.from_numpy(y_train)),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
    pin_memory=(device.type == "cuda"),
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_val_cont), torch.from_numpy(X_val_cat),
                  torch.from_numpy(y_val)),
    batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2,
)

# ---- Train ----
def train_nn(model, epochs=100, lr=1e-10, patience=5, resume=True):
    model = model.to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=2)
    crit = nn.MSELoss()
    start_epoch, best_val, hist = 0, float("inf"), {"train": [], "val": []}

    if resume and os.path.exists(CKPT):
        print(f"  ♻️ Resuming from {CKPT}")
        ckpt = torch.load(CKPT, map_location=device)
        model.load_state_dict(ckpt["model"])
        opt.load_state_dict(ckpt["optimizer"])
        start_epoch = ckpt["epoch"] + 1
        best_val = ckpt["best_val"]
        hist = ckpt["history"]
        print(f"  start={start_epoch}, best_val={best_val:.4f}")

    for epoch in range(start_epoch, epochs):
        model.train()
        tl, tn = 0.0, 0
        for c, k, y in train_loader:
            c, k, y = c.to(device), k.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(c, k), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl += loss.item()*y.size(0); tn += y.size(0)

        model.eval()
        vl, vn = 0.0, 0
        with torch.no_grad():
            for c, k, y in val_loader:
                c, k, y = c.to(device), k.to(device), y.to(device)
                vl += crit(model(c, k), y).item()*y.size(0); vn += y.size(0)
        tv = tl/tn; vv = vl/vn
        sched.step()
        hist["train"].append(tv); hist["val"].append(vv)
        print(f"  Epoch {epoch+1:2d}: train={tv:.4f} val_RMSLE={np.sqrt(vv):.4f}")

        if vv < best_val:
            best_val = vv
            torch.save({"model": model.state_dict(), "optimizer": opt.state_dict(),
                        "epoch": epoch, "best_val": best_val, "history": hist}, CKPT)
            print(f"     💾 saved")

        if len(hist["val"]) - int(np.argmin(hist["val"])) >= patience:
            print(f"  ⏹ early stop"); break

    ckpt = torch.load(CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    return model, hist

print("\n=== Training NN ===")
model = TabularNN(cont_dim=len(cont_indices), cat_specs=cat_specs)
print(f"  params: {sum(p.numel() for p in model.parameters()):,}")
t0 = time.time()
model, history = train_nn(model, epochs=30, lr=1e-3)
print(f"  total: {time.time()-t0:.1f}s")

# ---- Predict ----
def predict(m, c, k, bs=16384):
    m.eval()
    dl = DataLoader(TensorDataset(torch.from_numpy(c), torch.from_numpy(k)),
                     batch_size=bs, shuffle=False)
    out = []
    with torch.no_grad():
        for cb, kb in dl:
            out.append(m(cb.to(device), kb.to(device)).cpu().numpy())
    return np.concatenate(out)

print("\n=== Predicting ===")
nn_val_pred  = predict(model, X_val_cont, X_val_cat)
nn_test_pred = predict(model, X_test_cont, X_test_cat)

rmsle = float(np.sqrt(np.mean((y_val - nn_val_pred)**2)))
mae = float(np.mean(np.abs(y_val_sec - np.expm1(nn_val_pred))))
print(f"\n=== NN results ===")
print(f"  RMSLE = {rmsle:.4f}")
print(f"  MAE   = {mae:.1f}s")

np.save(f"{DRIVE_DIR}/nn_val_pred_log.npy",  nn_val_pred.astype(np.float32))
np.save(f"{DRIVE_DIR}/nn_test_pred_log.npy", nn_test_pred.astype(np.float32))
print(f"✅ saved to {DRIVE_DIR}/")

Device: cuda

=== Loading cache ===
  train: (1216551, 29) | val: (232722, 29) | test: (625134, 29)

=== Preprocessing ===
  cont: (1216551, 22) | cat: (1216551, 7)

=== Training NN ===
  params: 203,385
  ♻️ Resuming from /content/drive/MyDrive/nyc_taxi/nn_checkpoint.pt
  start=7, best_val=0.2021
  Epoch  8: train=0.2678 val_RMSLE=0.4776
  Epoch  9: train=0.2632 val_RMSLE=0.4679
  Epoch 10: train=0.2553 val_RMSLE=0.4523
  Epoch 11: train=0.2496 val_RMSLE=0.4598
  ⏹ early stop
  total: 70.8s

=== Predicting ===

=== NN results ===
  RMSLE = 0.4496
  MAE   = 361.9s
✅ saved to /content/drive/MyDrive/nyc_taxi/


In [11]:
# ============================================================
# CELL 6 — XGBoost + LightGBM
# ============================================================
import os, time
import numpy as np
import xgboost as xgb
import lightgbm as lgb

DRIVE_DIR = "/content/drive/MyDrive/nyc_taxi"
CACHE = f"{DRIVE_DIR}/cache.npz"

data = np.load(CACHE, allow_pickle=True)
X_train = data["X_train"]; X_val = data["X_val"]; X_test = data["X_test"]
y_train = data["y_train"]; y_val = data["y_val"]; y_val_sec = data["y_val_sec"]
test_ids = data["test_ids"]

# ---- XGBoost ----
print("=== XGBoost ===")
t0 = time.time()
dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)
dtest  = xgb.DMatrix(X_test)
bst = xgb.train(
    {"objective":"reg:squarederror","eval_metric":"rmse","eta":0.05,
     "max_depth":8,"tree_method":"hist","seed":42},
    dtrain, num_boost_round=2000, evals=[(dval,"val")],
    early_stopping_rounds=50, verbose_eval=100,
)
bi = bst.best_iteration + 1
xgb_val  = bst.predict(dval,  iteration_range=(0, bi))
xgb_test = bst.predict(dtest, iteration_range=(0, bi))
print(f"  RMSLE={float(np.sqrt(np.mean((y_val-xgb_val)**2))):.4f} | {time.time()-t0:.0f}s")
np.save(f"{DRIVE_DIR}/xgb_val_pred_log.npy",  xgb_val.astype(np.float32))
np.save(f"{DRIVE_DIR}/xgb_test_pred_log.npy", xgb_test.astype(np.float32))

# ---- LightGBM ----
print("\n=== LightGBM ===")
t0 = time.time()
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val   = lgb.Dataset(X_val,   label=y_val, reference=lgb_train)
gbm = lgb.train(
    {"objective":"regression","metric":"rmse","learning_rate":0.05,
     "num_leaves":63,"feature_fraction":0.8,"bagging_fraction":0.8,
     "verbose":-1,"seed":42},
    lgb_train, num_boost_round=2000, valid_sets=[lgb_val],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)],
)
lgb_val_p  = gbm.predict(X_val,  num_iteration=gbm.best_iteration)
lgb_test_p = gbm.predict(X_test, num_iteration=gbm.best_iteration)
print(f"  RMSLE={float(np.sqrt(np.mean((y_val-lgb_val_p)**2))):.4f} | {time.time()-t0:.0f}s")
np.save(f"{DRIVE_DIR}/lgbm_val_pred_log.npy",  lgb_val_p.astype(np.float32))
np.save(f"{DRIVE_DIR}/lgbm_test_pred_log.npy", lgb_test_p.astype(np.float32))

print("\n✅ Base models done")

=== XGBoost ===
[0]	val-rmse:0.75267
[100]	val-rmse:0.40650
[200]	val-rmse:0.39435
[300]	val-rmse:0.38799
[400]	val-rmse:0.38508
[500]	val-rmse:0.38303
[600]	val-rmse:0.38163
[700]	val-rmse:0.38032
[800]	val-rmse:0.37940
[900]	val-rmse:0.37836
[1000]	val-rmse:0.37770
[1100]	val-rmse:0.37710
[1200]	val-rmse:0.37648
[1300]	val-rmse:0.37604
[1400]	val-rmse:0.37569
[1500]	val-rmse:0.37543
[1600]	val-rmse:0.37526
[1700]	val-rmse:0.37495
[1800]	val-rmse:0.37459
[1900]	val-rmse:0.37444
[1999]	val-rmse:0.37423
  RMSLE=0.3742 | 352s

=== LightGBM ===
[100]	valid_0's rmse: 0.413103
[200]	valid_0's rmse: 0.400029
[300]	valid_0's rmse: 0.394986
[400]	valid_0's rmse: 0.39172
[500]	valid_0's rmse: 0.389679
[600]	valid_0's rmse: 0.387906
[700]	valid_0's rmse: 0.386094
[800]	valid_0's rmse: 0.384497
[900]	valid_0's rmse: 0.383456
[1000]	valid_0's rmse: 0.382687
[1100]	valid_0's rmse: 0.382073
[1200]	valid_0's rmse: 0.381422
[1300]	valid_0's rmse: 0.380709
[1400]	valid_0's rmse: 0.380196
[1500]	valid_0

In [13]:
# ============================================================
# Add CatBoost + retrain stacking
# ============================================================
from catboost import CatBoostRegressor
import numpy as np
import time

DRIVE_DIR = "/content/drive/MyDrive/nyc_taxi"
data = np.load(f"{DRIVE_DIR}/cache.npz", allow_pickle=True)
X_train, X_val, X_test = data["X_train"], data["X_val"], data["X_test"]
y_train, y_val = data["y_train"], data["y_val"]

print("=== CatBoost ===")
t0 = time.time()
cat = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3,
    loss_function="RMSE",
    early_stopping_rounds=50,
    random_seed=42,
    verbose=100,
    thread_count=-1,
)
cat.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

cat_val  = cat.predict(X_val)
cat_test = cat.predict(X_test)
rmsle = float(np.sqrt(np.mean((y_val - cat_val) ** 2)))
print(f"\nCatBoost RMSLE = {rmsle:.4f} | {time.time()-t0:.0f}s")

np.save(f"{DRIVE_DIR}/cat_val_pred_log.npy",  cat_val.astype(np.float32))
np.save(f"{DRIVE_DIR}/cat_test_pred_log.npy", cat_test.astype(np.float32))

=== CatBoost ===
0:	learn: 0.7340840	test: 0.7548375	best: 0.7548375 (0)	total: 282ms	remaining: 9m 24s
100:	learn: 0.4123914	test: 0.4236680	best: 0.4236680 (100)	total: 21.7s	remaining: 6m 47s
200:	learn: 0.3979835	test: 0.4085056	best: 0.4085056 (200)	total: 42.7s	remaining: 6m 21s
300:	learn: 0.3914868	test: 0.4022215	best: 0.4022215 (300)	total: 1m 2s	remaining: 5m 53s
400:	learn: 0.3871610	test: 0.3983926	best: 0.3983926 (400)	total: 1m 22s	remaining: 5m 29s
500:	learn: 0.3842543	test: 0.3959237	best: 0.3959237 (500)	total: 1m 42s	remaining: 5m 6s
600:	learn: 0.3816887	test: 0.3937890	best: 0.3937890 (600)	total: 2m 2s	remaining: 4m 44s
700:	learn: 0.3792613	test: 0.3918652	best: 0.3918652 (700)	total: 2m 22s	remaining: 4m 24s
800:	learn: 0.3774008	test: 0.3903988	best: 0.3903988 (800)	total: 2m 42s	remaining: 4m 3s
900:	learn: 0.3758676	test: 0.3892578	best: 0.3892578 (900)	total: 3m 2s	remaining: 3m 42s
1000:	learn: 0.3743876	test: 0.3881917	best: 0.3881917 (1000)	total: 3m 23s

In [12]:
# ============================================================
# CELL 7 — Stacking + Submission
# ============================================================
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, LinearRegression
from scipy.optimize import minimize

DRIVE_DIR = "/content/drive/MyDrive/nyc_taxi"
data = np.load(f"{DRIVE_DIR}/cache.npz", allow_pickle=True)
y_val = data["y_val"]; y_val_sec = data["y_val_sec"]; test_ids = data["test_ids"]

val_preds, test_preds = {}, {}
for name, key in [("NN","nn"),("XGBoost","xgb"),("LightGBM","lgbm")]:
    vp = f"{DRIVE_DIR}/{key}_val_pred_log.npy"
    tp = f"{DRIVE_DIR}/{key}_test_pred_log.npy"
    if os.path.exists(vp):
        val_preds[name]  = np.load(vp)
        test_preds[name] = np.load(tp)
        print(f"  {name:10s} RMSLE={float(np.sqrt(np.mean((y_val-val_preds[name])**2))):.4f}")

names = list(val_preds.keys())
V = np.column_stack([val_preds[m]  for m in names])
T = np.column_stack([test_preds[m] for m in names])

def rmsle(p): return float(np.sqrt(np.mean((y_val-p)**2)))
strategies = {"simple_avg": V.mean(axis=1)}

def wl(w):
    wa = np.abs(w); return rmsle(V @ (wa/(wa.sum()+1e-9)))
best_w, best_l = None, 1e9
for _ in range(10):
    r = minimize(wl, np.random.rand(len(names))+0.1, method="Nelder-Mead",
                 options={"xatol":1e-6,"fatol":1e-8,"maxiter":500})
    if r.fun < best_l:
        best_l = r.fun; wa = np.abs(r.x); best_w = wa/wa.sum()
strategies["weighted"] = V @ best_w
print(f"  weights: {dict(zip(names, np.round(best_w, 3)))}")

for a in [0.01, 0.1, 1.0, 10.0]:
    strategies[f"ridge_a{a}"] = Ridge(alpha=a).fit(V, y_val).predict(V)
strategies["nnls"] = LinearRegression(positive=True).fit(V, y_val).predict(V)

df = pd.DataFrame([{"strategy": n, "rmsle": rmsle(p)} for n, p in strategies.items()])
df = df.sort_values("rmsle").reset_index(drop=True)
print(f"\n=== Stacking results ===")
print(df.round(4).to_string(index=False))

best = df.iloc[0]["strategy"]
print(f"\n🏆 Best: {best}")

if best == "simple_avg": ft = T.mean(axis=1)
elif best == "weighted":  ft = T @ best_w
elif best.startswith("ridge_a"):
    a = float(best.split("_a")[1])
    ft = Ridge(alpha=a).fit(V, y_val).predict(T)
elif best == "nnls":
    ft = LinearRegression(positive=True).fit(V, y_val).predict(T)

final = np.clip(np.expm1(ft), 1.0, 86400.0)
sub = pd.DataFrame({"id": test_ids, "trip_duration": final})
sub.to_csv(f"{DRIVE_DIR}/submission.csv", index=False)

print(f"\n✅ Submission: {DRIVE_DIR}/submission.csv")
print(f"   Rows: {len(sub):,}")
print(f"\nFinal stats:")
print(sub["trip_duration"].describe().round(1).to_string())

from google.colab import files
#files.download(f"{DRIVE_DIR}/submission.csv")

  NN         RMSLE=0.4498
  XGBoost    RMSLE=0.3742
  LightGBM   RMSLE=0.3776
  weights: {'NN': np.float64(0.0), 'XGBoost': np.float64(0.714), 'LightGBM': np.float64(0.286)}

=== Stacking results ===
   strategy  rmsle
ridge_a0.01 0.3734
 ridge_a0.1 0.3734
 ridge_a1.0 0.3734
ridge_a10.0 0.3734
       nnls 0.3736
   weighted 0.3736
 simple_avg 0.3852

🏆 Best: ridge_a0.01

✅ Submission: /content/drive/MyDrive/nyc_taxi/submission.csv
   Rows: 625,134

Final stats:
count    625134.0
mean        808.6
std         592.3
min          16.3
25%         401.9
50%         649.4
75%        1033.4
max        6836.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, LinearRegression

DRIVE_DIR = "/content/drive/MyDrive/nyc_taxi"
data = np.load(f"{DRIVE_DIR}/cache.npz", allow_pickle=True)
y_val = data["y_val"]; y_val_sec = data["y_val_sec"]; test_ids = data["test_ids"]

# Load tree models only (NN excluded)
val_preds, test_preds = {}, {}
for name, key in [("XGBoost","xgb"), ("LightGBM","lgbm"), ("CatBoost","cat")]:
    if os.path.exists(f"{DRIVE_DIR}/{key}_val_pred_log.npy"):
        val_preds[name]  = np.load(f"{DRIVE_DIR}/{key}_val_pred_log.npy")
        test_preds[name] = np.load(f"{DRIVE_DIR}/{key}_test_pred_log.npy")
        print(f"  {name:10s} RMSLE={float(np.sqrt(np.mean((y_val-val_preds[name])**2))):.4f}")

names = list(val_preds.keys())
V = np.column_stack([val_preds[m]  for m in names])
T = np.column_stack([test_preds[m] for m in names])

def rmsle(p): return float(np.sqrt(np.mean((y_val-p)**2)))

strategies = {"simple_avg": V.mean(axis=1)}
for a in [0.01, 0.1, 1.0, 10.0]:
    strategies[f"ridge_a{a}"] = Ridge(alpha=a).fit(V, y_val).predict(V)
strategies["nnls"] = LinearRegression(positive=True).fit(V, y_val).predict(V)

df = pd.DataFrame([{"strategy": n, "rmsle": rmsle(p)} for n, p in strategies.items()])
df = df.sort_values("rmsle").reset_index(drop=True)
print("\n=== Stacking results ===")
print(df.round(4).to_string(index=False))

best = df.iloc[0]["strategy"]
if best == "simple_avg": ft = T.mean(axis=1)
elif best.startswith("ridge_a"):
    a = float(best.split("_a")[1])
    ft = Ridge(alpha=a).fit(V, y_val).predict(T)
elif best == "nnls":
    ft = LinearRegression(positive=True).fit(V, y_val).predict(T)

final = np.clip(np.expm1(ft), 1.0, 86400.0)
sub = pd.DataFrame({"id": test_ids, "trip_duration": final})
sub.to_csv(f"{DRIVE_DIR}/submission_v2.csv", index=False)
print(f"\n✅ Saved: {DRIVE_DIR}/submission_v2.csv")

  XGBoost    RMSLE=0.3742
  LightGBM   RMSLE=0.3776
  CatBoost   RMSLE=0.3820

=== Stacking results ===
   strategy  rmsle
ridge_a0.01 0.3735
 ridge_a0.1 0.3735
 ridge_a1.0 0.3735
ridge_a10.0 0.3735
       nnls 0.3736
 simple_avg 0.3753

✅ Saved: /content/drive/MyDrive/nyc_taxi/submission_v2.csv
